# 01 — Backbone forward 검증

**목표**:
- ResNet-18/50 backbone이 정상 동작하는지
- 96x96 입력 (STL10) + 32x32 입력 (CIFAR10) 모두 처리 가능한지
- feature dim이 예상대로 나오는지 (18 → 512, 50 → 2048)
- small_image 옵션 효과 비교

In [ ]:
%load_ext autoreload
%autoreload 2

import torch
from ssl_lib.models.backbone import ResNetBackbone

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

## Cell 2 — ResNet-18 + small_image 검증

In [ ]:
bb18 = ResNetBackbone(name='resnet18', small_image=True).to(device)
print(f'ResNet-18 small_image=True: feature_dim={bb18.feature_dim}')
n_params = sum(p.numel() for p in bb18.parameters())
print(f'params: {n_params/1e6:.2f}M')

# 96x96 입력
x96 = torch.randn(2, 3, 96, 96, device=device)
feat96 = bb18(x96)
print(f'96x96 input → feature: {feat96.shape}')  # (2, 512)

# 32x32 입력 (CIFAR10)
x32 = torch.randn(2, 3, 32, 32, device=device)
feat32 = bb18(x32)
print(f'32x32 input → feature: {feat32.shape}')  # (2, 512)

# 224x224 입력 (혹시 evaluate.py가 이걸 쓸 수도)
x224 = torch.randn(2, 3, 224, 224, device=device)
feat224 = bb18(x224)
print(f'224x224 input → feature: {feat224.shape}')

## Cell 3 — ResNet-50 + small_image 검증

In [ ]:
bb50 = ResNetBackbone(name='resnet50', small_image=True).to(device)
print(f'ResNet-50 small_image=True: feature_dim={bb50.feature_dim}')
n_params = sum(p.numel() for p in bb50.parameters())
print(f'params: {n_params/1e6:.2f}M')

x96 = torch.randn(2, 3, 96, 96, device=device)
feat = bb50(x96)
print(f'96x96 input → feature: {feat.shape}')  # (2, 2048)

## Cell 4 — small_image=False vs True 비교

원본 ResNet (small_image=False)은 첫 conv가 7x7/stride2 + maxpool이라
96x96 → 매우 작은 feature map → 정보 손실 큼.
small_image=True가 feature 보존에 유리.

In [ ]:
bb_full = ResNetBackbone(name='resnet50', small_image=False).to(device)
bb_small = ResNetBackbone(name='resnet50', small_image=True).to(device)

# 내부 layer 확인 (첫 conv)
print('small_image=False (original ResNet):')
print(f'  conv1: {bb_full.net.conv1}')
print(f'  maxpool: {bb_full.net.maxpool}')
print()
print('small_image=True:')
print(f'  conv1: {bb_small.net.conv1}')
print(f'  maxpool: {bb_small.net.maxpool}')

## Cell 5 — Forward 시간 측정 (sanity)

In [ ]:
import time

for model_name, m in [('R-18', bb18), ('R-50', bb50)]:
    m.eval()
    x = torch.randn(64, 3, 96, 96, device=device)
    # warmup
    for _ in range(3):
        with torch.no_grad():
            _ = m(x)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    t = time.time()
    for _ in range(10):
        with torch.no_grad():
            _ = m(x)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    print(f'{model_name}: {(time.time()-t)*100:.2f} ms/batch (batch=64)')

## ✅ 체크리스트

- [ ] ResNet-18 feature_dim = 512
- [ ] ResNet-50 feature_dim = 2048
- [ ] 96x96 / 32x32 / 224x224 모든 해상도에서 forward OK
- [ ] OOM 없음

**참고**: evaluate.py를 받으면, evaluate가 어떤 입력 해상도와 어떤 호출 방식을 쓰는지 보고
small_image 옵션을 조정해야 함.